# Syntopy Between Scales - Master Thesis

## Intial Data Preparation

In [1]:
import ee
import geemap
import pandas as pd
import geopandas as gpd

In [3]:
#  Initialise Earth Engine Project ID
try:
    ee.Initialize(project='syntopypredictors2026')

except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='syntopypredictors2026')

In [4]:
connectivity_test = ee.Number(1).add(2).getInfo()
print('connectivity test: ', connectivity_test)

connectivity test:  3


In [108]:
dat = gpd.read_file('route_dat.gpkg', engine = 'fiona')
dat['lon'] = dat.geometry.x
dat['lat'] = dat.geometry.y
dat.head()

,RTENO,Year,geometry,lon,lat
0,12404002,2017,POINT (-111.4102 49.87324),-111.41020,49.87324
1,12404002,2018,POINT (-111.4102 49.87324),-111.41020,49.87324
2,12404002,2019,POINT (-111.4102 49.87324),-111.41020,49.87324
3,12404003,2017,POINT (-112.39699 49.08995),-112.39699,49.08995
4,12404004,2017,POINT (-113.56982 49.81509),-113.56982,49.81509


In [109]:
# Keep only the unique routes based on RTENO
unique_routes = dat.drop_duplicates(subset=['RTENO']).reset_index(drop=True)

print(f'Original rows with multi-year records: {len(dat)}')
print(f'Unique routes to extract: {len(unique_routes)}')

Original rows with multi-year records: 8914
Unique routes to extract: 3614


In [111]:
import time
import pandas as pd

chunk_size = 500
all_results = []

print("Starting batched local extraction...")

for i in range(0, len(unique_routes), chunk_size):
  chunk = unique_routes.iloc[i : i + chunk_size]

  # 1. Build features for the current chunk
  features = []
  for _, row in chunk.iterrows():
    point = ee.Geometry.Point([row["lon"], row["lat"]])
    buff = point.buffer(3000)  # 3000m buffer
    feat = ee.Feature(buff, {"RTENO": row["RTENO"]})
    features.append(feat)

  points_fc = ee.FeatureCollection(features)

  # 2. Run reduction on the small batch
  sampled_chunk = ndvi.reduceRegions(
      collection=points_fc, reducer=ee.Reducer.mean(), scale=30
  )

  # 3. Safely fetch info for this small batch
  try:
    info = sampled_chunk.getInfo()
    for f in info["features"]:
      all_results.append(f["properties"])
    print(
        f"Successfully processed rows {i} to"
        f" {min(i + chunk_size, len(dat))}"
    )
  except Exception as e:
    print(f"Error on chunk starting at {i}: {e}")

  # Brief pause to be kind to the server
  time.sleep(1)

# Combine all chunks back into a single DataFrame
df_ndvi = pd.DataFrame(all_results)
dat_final = unique_routes.merge(df_ndvi[["RTENO", "mean"]], on="RTENO", how="left")
print("All routes successfully extracted and merged!")

Starting batched local extraction...
Successfully processed rows 0 to 500
Successfully processed rows 500 to 1000
Successfully processed rows 1000 to 1500
Successfully processed rows 1500 to 2000
Successfully processed rows 2000 to 2500
Successfully processed rows 2500 to 3000
Successfully processed rows 3000 to 3500
Successfully processed rows 3500 to 4000
All routes successfully extracted and merged!


In [ ]:
dat_final['NDVI_mean_3km'] = dat_final['mean']
dat_final.head()

In [122]:
dat_final.to_file("NDVI_mean_3km_buffer.gpkg", driver='GPKG')

In [117]:
m = dat_final.explore(
    column = 'mean',
    cmap='RdYlGn',
    legend=True,
    tooltip=['mean'],
    popup=True)
m